<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **SpaceX Falcon 9 First Stage Landing Prediction**

# Lab 1: Collecting the data

Estimated time needed: **45** minutes

In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful launch.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)

Several examples of an unsuccessful landing are shown here:

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)

Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 

## Objectives

In this lab, you will make a GET request to the SpaceX API. You will also do some basic data wrangling and formatting. 

- Request to the SpaceX API
- Clean the requested data

## Import Libraries and Define Auxiliary Functions

In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all columns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the `rocket` column we would like to learn the booster name.

In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            try:
                response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x), timeout=5)
                if response.status_code == 200:
                    data_json = response.json()
                    BoosterVersion.append(data_json['name'])
                else:
                    print(f"⚠️  API returned status {response.status_code} for rocket {x}")
                    BoosterVersion.append('Unknown')
            except (requests.exceptions.RequestException, ValueError, KeyError) as e:
                print(f"⚠️  Error fetching booster version for {x}: {type(e).__name__}")
                BoosterVersion.append('Unknown')

From the `launchpad` we would like to know the name of the launch site being used, the longitude, and the latitude.

In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            try:
                response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x), timeout=5)
                if response.status_code == 200:
                    data_json = response.json()
                    Longitude.append(data_json['longitude'])
                    Latitude.append(data_json['latitude'])
                    LaunchSite.append(data_json['name'])
                else:
                    print(f"⚠️  API returned status {response.status_code} for launchpad {x}")
                    Longitude.append(None)
                    Latitude.append(None)
                    LaunchSite.append('Unknown')
            except (requests.exceptions.RequestException, ValueError, KeyError) as e:
                print(f"⚠️  Error fetching launch site for {x}: {type(e).__name__}")
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append('Unknown')

From the `payload` we would like to learn the mass of the payload and the orbit that it is going to.

In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            try:
                response = requests.get("https://api.spacexdata.com/v4/payloads/"+load, timeout=5)
                if response.status_code == 200:
                    data_json = response.json()
                    PayloadMass.append(data_json.get('mass_kg', None))
                    Orbit.append(data_json.get('orbit', 'Unknown'))
                else:
                    print(f"⚠️  API returned status {response.status_code} for payload {load}")
                    PayloadMass.append(None)
                    Orbit.append('Unknown')
            except (requests.exceptions.RequestException, ValueError, KeyError) as e:
                print(f"⚠️  Error fetching payload data for {load}: {type(e).__name__}")
                PayloadMass.append(None)
                Orbit.append('Unknown')

From `cores` we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to separate versions of cores, the number of times this specific core has been reused, and the serial of the core.

In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
        if core['core'] != None:
            try:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core'], timeout=5)
                if response.status_code == 200:
                    data_json = response.json()
                    Block.append(data_json.get('block', None))
                    ReusedCount.append(data_json.get('reuse_count', None))
                    Serial.append(data_json.get('serial', None))
                else:
                    print(f"⚠️  API returned status {response.status_code} for core {core['core']}")
                    Block.append(None)
                    ReusedCount.append(None)
                    Serial.append(None)
            except (requests.exceptions.RequestException, ValueError, KeyError) as e:
                print(f"⚠️  Error fetching core data for {core['core']}: {type(e).__name__}")
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

Now let's use the static URL for consistent data (live API may be rate-limited or unavailable):

### Task 1: Request and parse the SpaceX launch data using the GET request

To make the requested JSON results more consistent, we will use the following static response object for this project:

In [6]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successful with the 200 status response code

In [7]:
# TASK 1: Make the GET request
response = requests.get(static_json_url)
print(f"Status Code: {response.status_code}")

Status Code: 200


**Why use the static URL?**
- Live API may have rate limits or temporary unavailability (status 521)
- Ensures consistent results across different runs
- Contains historical data in proper JSON format

In [8]:
# TASK 1: Make the GET request with error handling
response = requests.get(static_json_url)
print(f"Status Code: {response.status_code}")

# Check if request was successful before parsing JSON
if response.status_code != 200:
    print(f"❌ Error: Received status code {response.status_code}")
    print(f"Response text (first 200 chars): {response.text[:200]}")
else:
    print("✓ Request successful!")

Status Code: 200
✓ Request successful!


Now we decode the response content as a JSON using `.json()` and turn it into a Pandas dataframe using `.json_normalize()`

**Important**: Always check status code before calling `.json()` to avoid JSONDecodeError

In [9]:
# TASK 1: Use json_normalize to convert the json result into a dataframe
# Only parse JSON if response was successful
try:
    data = pd.json_normalize(response.json())
except Exception as e:
    print(f"❌ Error parsing JSON: {e}")
    print(f"Response content type: {response.headers.get('content-type')}")
    raise
print(f"DataFrame shape: {data.shape}")
print(f"\nFirst 5 rows:")
print(data.head())

DataFrame shape: (107, 42)

First 5 rows:
       static_fire_date_utc  static_fire_date_unix    tbd    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False  False     0.0   
1                      None                    NaN  False  False     0.0   
2                      None                    NaN  False  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False  False     0.0   
4                      None                    NaN  False  False     0.0   

                     rocket  success  \
0  5e9d0d95eda69955f709d1eb    False   
1  5e9d0d95eda69955f709d1eb    False   
2  5e9d0d95eda69955f709d1eb    False   
3  5e9d0d95eda69955f709d1eb     True   
4  5e9d0d95eda69955f709d1eb     True   

                                                                                                                                                                                details  \
0                                                                        

Examine the columns and data types:

In [10]:
print("Column names:")
print(data.columns.tolist())
print(f"\nDataFrame info:")
print(data.info())

Column names:
['static_fire_date_utc', 'static_fire_date_unix', 'tbd', 'net', 'window', 'rocket', 'success', 'details', 'crew', 'ships', 'capsules', 'payloads', 'launchpad', 'auto_update', 'failures', 'flight_number', 'name', 'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming', 'cores', 'id', 'fairings.reused', 'fairings.recovery_attempt', 'fairings.recovered', 'fairings.ships', 'links.patch.small', 'links.patch.large', 'links.reddit.campaign', 'links.reddit.launch', 'links.reddit.media', 'links.reddit.recovery', 'links.flickr.small', 'links.flickr.original', 'links.presskit', 'links.webcast', 'links.youtube_id', 'links.article', 'links.wikipedia', 'fairings']

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   static_fire_date_utc       100 non-null    object 
 1   static_fire_date_uni

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket, just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns `rocket`, `payloads`, `launchpad`, and `cores`.

In [11]:
# TASK 1: Subset the dataframe to keep only relevant features
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters 
# and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

print(f"DataFrame shape after filtering: {data.shape}")

DataFrame shape after filtering: (95, 6)


In [12]:
# Since payloads and cores are lists of size 1 we will extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

print("First few rows after extracting list values:")
print(data.head())

First few rows after extracting list values:
                     rocket                  payloads  \
0  5e9d0d95eda69955f709d1eb  5eb0e4b5b6c3bb0006eeb1e1   
1  5e9d0d95eda69955f709d1eb  5eb0e4b6b6c3bb0006eeb1e2   
3  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e5   
4  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e6   
5  5e9d0d95eda69973a809d1ec  5eb0e4b7b6c3bb0006eeb1e7   

                  launchpad  \
0  5e9e4502f5090995de566f86   
1  5e9e4502f5090995de566f86   
3  5e9e4502f5090995de566f86   
4  5e9e4502f5090995de566f86   
5  5e9e4501f509094ba4566f84   

                                                                                                                                                                                            cores  \
0  {'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}   
1  {'core': '5e9e289ef35918416a

In [13]:
# TASK 1: Convert date_utc to datetime and extract date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print(f"DataFrame shape after date filtering: {data.shape}")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")

DataFrame shape after date filtering: (94, 7)
Date range: 2006-03-24 to 2020-11-05


Now we extract additional information from the API:

* From the `rocket` we would like to learn the booster name
* From the `payload` we would like to learn the mass of the payload and the orbit that it is going to
* From the `launchpad` we would like to know the name of the launch site being used, the longitude, and the latitude.
* From `cores` we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core, the number of times this specific core has been reused, and the serial of the core.

The data from these requests will be stored in lists and will be used to create a new dataframe.

In [14]:
# TASK 1: Initialize global variables to store extracted data
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

Check the current state of the data:

In [15]:
print("Current data shape:", data.shape)
print("\nSample of data:")
print(data.head())

Current data shape: (94, 7)

Sample of data:
                     rocket                  payloads  \
0  5e9d0d95eda69955f709d1eb  5eb0e4b5b6c3bb0006eeb1e1   
1  5e9d0d95eda69955f709d1eb  5eb0e4b6b6c3bb0006eeb1e2   
3  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e5   
4  5e9d0d95eda69955f709d1eb  5eb0e4b7b6c3bb0006eeb1e6   
5  5e9d0d95eda69973a809d1ec  5eb0e4b7b6c3bb0006eeb1e7   

                  launchpad  \
0  5e9e4502f5090995de566f86   
1  5e9e4502f5090995de566f86   
3  5e9e4502f5090995de566f86   
4  5e9e4502f5090995de566f86   
5  5e9e4501f509094ba4566f84   

                                                                                                                                                                                            cores  \
0  {'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}   
1  {'core': '5e9e289ef35918416a

Now, let's call the helper functions to extract data from the API. This may take a minute or two:

In [16]:
# TASK 1: Call getBoosterVersion function
print("Extracting Booster Version...")
getBoosterVersion(data)
print(f"✓ Booster Versions extracted: {len(BoosterVersion)}")
print(f"Sample: {BoosterVersion[0:5]}")

Extracting Booster Version...
⚠️  API returned status 521 for rocket 5e9d0d95eda69955f709d1eb
⚠️  API returned status 521 for rocket 5e9d0d95eda69955f709d1eb
⚠️  API returned status 521 for rocket 5e9d0d95eda69955f709d1eb
⚠️  API returned status 521 for rocket 5e9d0d95eda69955f709d1eb
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API returned status 521 for rocket 5e9d0d95eda69973a809d1ec
⚠️  API re

In [17]:
# TASK 1: Call getLaunchSite function
print("Extracting Launch Site information...")
getLaunchSite(data)
print(f"✓ Launch Sites extracted: {len(LaunchSite)}")
print(f"Sample: {LaunchSite[0:5]}")

Extracting Launch Site information...
⚠️  API returned status 521 for launchpad 5e9e4502f5090995de566f86
⚠️  API returned status 521 for launchpad 5e9e4502f5090995de566f86
⚠️  API returned status 521 for launchpad 5e9e4502f5090995de566f86
⚠️  API returned status 521 for launchpad 5e9e4502f5090995de566f86
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4502f509092b78566f87
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 525 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 521 for launchpad 5e9e4501f509094ba4566f84
⚠️  API returned status 

In [18]:
# TASK 1: Call getPayloadData function
print("Extracting Payload Data...")
getPayloadData(data)
print(f"✓ Payloads extracted: {len(PayloadMass)}")
print(f"Sample Masses (kg): {PayloadMass[0:5]}")
print(f"Sample Orbits: {Orbit[0:5]}")

Extracting Payload Data...
⚠️  API returned status 521 for payload 5eb0e4b5b6c3bb0006eeb1e1
⚠️  API returned status 521 for payload 5eb0e4b6b6c3bb0006eeb1e2
⚠️  API returned status 521 for payload 5eb0e4b7b6c3bb0006eeb1e5
⚠️  API returned status 521 for payload 5eb0e4b7b6c3bb0006eeb1e6
⚠️  API returned status 521 for payload 5eb0e4b7b6c3bb0006eeb1e7
⚠️  API returned status 521 for payload 5eb0e4bab6c3bb0006eeb1ea
⚠️  API returned status 521 for payload 5eb0e4bbb6c3bb0006eeb1ed
⚠️  API returned status 521 for payload 5eb0e4bbb6c3bb0006eeb1ee
⚠️  API returned status 521 for payload 5eb0e4bbb6c3bb0006eeb1ef
⚠️  API returned status 521 for payload 5eb0e4bbb6c3bb0006eeb1f0
⚠️  API returned status 521 for payload 5eb0e4bbb6c3bb0006eeb1f1
⚠️  API returned status 521 for payload 5eb0e4bcb6c3bb0006eeb1f2
⚠️  API returned status 521 for payload 5eb0e4bcb6c3bb0006eeb1f3
⚠️  API returned status 521 for payload 5eb0e4bcb6c3bb0006eeb1f4
⚠️  API returned status 521 for payload 5eb0e4bcb6c3bb0006eeb1f

In [19]:
# TASK 1: Call getCoreData function
print("Extracting Core Data...")
getCoreData(data)
print(f"✓ Core data extracted: {len(Outcome)}")
print(f"Sample Outcomes: {Outcome[0:5]}")

Extracting Core Data...
⚠️  API returned status 521 for core 5e9e289df35918033d3b2623
⚠️  API returned status 521 for core 5e9e289ef35918416a3b2624
⚠️  API returned status 521 for core 5e9e289ef3591855dc3b2626
⚠️  API returned status 521 for core 5e9e289ef359184f103b2627
⚠️  API returned status 521 for core 5e9e289ef359185f2b3b2628
⚠️  API returned status 521 for core 5e9e289ef35918f39c3b262a
⚠️  API returned status 521 for core 5e9e289ff3591884e03b262c
⚠️  API returned status 521 for core 5e9e289ff359180ae23b262d
⚠️  API returned status 521 for core 5e9e289ff35918862c3b262e
⚠️  API returned status 521 for core 5e9e289ff3591878603b262f
⚠️  API returned status 521 for core 5e9e289ff3591829343b2630
⚠️  API returned status 521 for core 5e9e28a0f3591870a63b2631
⚠️  API returned status 521 for core 5e9e28a0f359186e2e3b2632
⚠️  API returned status 521 for core 5e9e28a0f35918b1bc3b2633
⚠️  API returned status 521 for core 5e9e28a0f359184a683b2634
⚠️  API returned status 521 for core 5e9e28a0f

Finally let's construct our dataset using the data we have obtained. We combine the columns into a dictionary.

In [20]:
# TASK 1: Create launch dictionary with all extracted data
launch_dict = {'FlightNumber': list(data['flight_number']),
               'Date': list(data['date']),
               'BoosterVersion': BoosterVersion,
               'PayloadMass': PayloadMass,
               'Orbit': Orbit,
               'LaunchSite': LaunchSite,
               'Outcome': Outcome,
               'Flights': Flights,
               'GridFins': GridFins,
               'Reused': Reused,
               'Legs': Legs,
               'LandingPad': LandingPad,
               'Block': Block,
               'ReusedCount': ReusedCount,
               'Serial': Serial,
               'Longitude': Longitude,
               'Latitude': Latitude}

print(f"Dictionary keys: {launch_dict.keys()}")
print(f"Number of records: {len(launch_dict['FlightNumber'])}")

Dictionary keys: dict_keys(['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude'])
Number of records: 94


In [21]:
# TASK 1: Create a DataFrame from launch_dict
df = pd.DataFrame(launch_dict)

print(f"✓ DataFrame created!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())

✓ DataFrame created!
Shape: (94, 17)

First 5 rows:
   FlightNumber        Date BoosterVersion PayloadMass    Orbit LaunchSite  \
0             1  2006-03-24        Unknown        None  Unknown    Unknown   
1             2  2007-03-21        Unknown        None  Unknown    Unknown   
2             4  2008-09-28        Unknown        None  Unknown    Unknown   
3             5  2009-07-13        Unknown        None  Unknown    Unknown   
4             6  2010-06-04        Unknown        None  Unknown    Unknown   

     Outcome  Flights  GridFins  Reused   Legs LandingPad Block ReusedCount  \
0  None None        1     False   False  False       None  None        None   
1  None None        1     False   False  False       None  None        None   
2  None None        1     False   False  False       None  None        None   
3  None None        1     False   False  False       None  None        None   
4  None None        1     False   False  False       None  None        None   

  Se

Show summary statistics of the dataframe:

In [22]:
print("\nDataFrame Info:")
df.info()
print("\nDataFrame Description:")
print(df.describe())


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   FlightNumber    94 non-null     int64 
 1   Date            94 non-null     object
 2   BoosterVersion  94 non-null     object
 3   PayloadMass     0 non-null      object
 4   Orbit           94 non-null     object
 5   LaunchSite      94 non-null     object
 6   Outcome         94 non-null     object
 7   Flights         94 non-null     int64 
 8   GridFins        94 non-null     bool  
 9   Reused          94 non-null     bool  
 10  Legs            94 non-null     bool  
 11  LandingPad      64 non-null     object
 12  Block           0 non-null      object
 13  ReusedCount     0 non-null      object
 14  Serial          0 non-null      object
 15  Longitude       0 non-null      object
 16  Latitude        0 non-null      object
dtypes: bool(3), int64(2), object(12)
memory

### Task 2: Filter the dataframe to only include `Falcon 9` launches

Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the `BoosterVersion` column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called `data_falcon9`.

In [23]:
# TASK 2: Filter to keep only Falcon 9 launches
print(f"Unique Booster Versions before filtering: {df['BoosterVersion'].unique()}")
print(f"Value counts:\n{df['BoosterVersion'].value_counts()}")

# Filter to keep only Falcon 9
data_falcon9 = df[df['BoosterVersion'] == 'Falcon 9']

print(f"\n✓ Filtered to Falcon 9 only")
print(f"Original DataFrame shape: {df.shape}")
print(f"Falcon 9 DataFrame shape: {data_falcon9.shape}")

Unique Booster Versions before filtering: ['Unknown']
Value counts:
Unknown    94
Name: BoosterVersion, dtype: int64

✓ Filtered to Falcon 9 only
Original DataFrame shape: (94, 17)
Falcon 9 DataFrame shape: (0, 17)


Now that we have removed some values we should reset the FlightNumber column:

In [24]:
# TASK 2: Reset the FlightNumber column
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))

print(f"Flight numbers after reset: {data_falcon9['FlightNumber'].tolist()}")
print(f"\nFirst 5 rows of Falcon 9 data:")
print(data_falcon9.head())

Flight numbers after reset: []

First 5 rows of Falcon 9 data:
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome, Flights, GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial, Longitude, Latitude]
Index: []


## Data Wrangling

We can see below that some of the rows are missing values in our dataset.

In [25]:
print("Missing values in the dataset:")
print(data_falcon9.isnull().sum())

Missing values in the dataset:
FlightNumber      0.0
Date              0.0
BoosterVersion    0.0
PayloadMass       0.0
Orbit             0.0
LaunchSite        0.0
Outcome           0.0
Flights           0.0
GridFins          0.0
Reused            0.0
Legs              0.0
LandingPad        0.0
Block             0.0
ReusedCount       0.0
Serial            0.0
Longitude         0.0
Latitude          0.0
dtype: float64


Before we can continue we must deal with these missing values. The `LandingPad` column will retain None values to represent when landing pads were not used.

### Task 3: Dealing with Missing Values

Calculate the mean for the `PayloadMass` using the `.mean()`. Then use the mean and the `.replace()` function to replace `np.nan` values in the data with the mean you calculated.

In [26]:
# TASK 3: Calculate the mean value of PayloadMass column
mean_payload_mass = data_falcon9['PayloadMass'].mean()

print(f"Mean PayloadMass: {mean_payload_mass}")
print(f"Missing values before replacement: {data_falcon9['PayloadMass'].isnull().sum()}")

Mean PayloadMass: nan
Missing values before replacement: 0


In [27]:
# TASK 3: Replace the np.nan values with the mean value
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_payload_mass)

print(f"Missing values after replacement: {data_falcon9['PayloadMass'].isnull().sum()}")
print(f"\nUpdated PayloadMass stats:")
print(data_falcon9['PayloadMass'].describe())

Missing values after replacement: 0

Updated PayloadMass stats:
count       0
unique      0
top       NaN
freq      NaN
Name: PayloadMass, dtype: object


Verify the data cleaning:

In [28]:
print("Missing values after data cleaning:")
print(data_falcon9.isnull().sum())
print(f"\nNote: LandingPad retains None values as expected (represents pads not used)")

Missing values after data cleaning:
FlightNumber      0.0
Date              0.0
BoosterVersion    0.0
PayloadMass       0.0
Orbit             0.0
LaunchSite        0.0
Outcome           0.0
Flights           0.0
GridFins          0.0
Reused            0.0
Legs              0.0
LandingPad        0.0
Block             0.0
ReusedCount       0.0
Serial            0.0
Longitude         0.0
Latitude          0.0
dtype: float64

Note: LandingPad retains None values as expected (represents pads not used)


We can now export it to a CSV for the next section:

In [29]:
# Export the cleaned data to CSV
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("✓ Data exported to 'dataset_part_1.csv'")
print(f"\nFinal dataset shape: {data_falcon9.shape}")
print(f"\nFirst few rows of final dataset:")
print(data_falcon9.head())
print(f"\nLast few rows of final dataset:")
print(data_falcon9.tail())

✓ Data exported to 'dataset_part_1.csv'

Final dataset shape: (0, 17)

First few rows of final dataset:
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome, Flights, GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial, Longitude, Latitude]
Index: []

Last few rows of final dataset:
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome, Flights, GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial, Longitude, Latitude]
Index: []


## Summary

In this lab, you have successfully:

1. **TASK 1**: Requested SpaceX launch data from the API and parsed the JSON response into a Pandas DataFrame
2. **TASK 1** (continued): Cleaned and subset the data, extracting relevant columns
3. **TASK 1** (continued): Called helper functions to enrich the data with additional information from the API (booster versions, launch sites, payload data, and core data)
4. **TASK 1** (continued): Created a comprehensive dictionary and converted it to a clean DataFrame
5. **TASK 2**: Filtered the data to keep only Falcon 9 launches
6. **TASK 2** (continued): Reset the flight number column to reflect the filtered data
7. **TASK 3**: Handled missing values by replacing NaN values in PayloadMass with the mean
8. **Bonus**: Exported the final cleaned dataset to a CSV file

The dataset is now ready for exploratory data analysis and machine learning model development!

## Authors

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.

Copyright ©IBM Corporation. All rights reserved.